# PyTorch basics

Every neural network, from a 3-layer classifier to a 400B-parameter LLM, is built from three things you'll learn here:

1. Tensors, which are n-dimensional arrays of numbers with shapes you have to track constantly.
2. Autograd, where PyTorch records the operations you do and computes gradients automatically.
3. The training loop: forward pass, loss, backward pass, update, repeat.

Run cells in order. Before you run a cell that prints a shape, predict the shape first. Shape intuition is the skill that separates people who can debug models from people who can't.

In [ ]:
import time
import torch

print("PyTorch", torch.__version__)

# Pick the fastest device available. "mps" is Apple's GPU backend. "cuda" is NVIDIA (you'd see it on a rented GPU).
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

_ = torch.manual_seed(42)  # makes random numbers repeatable

## 1. Creating tensors

A tensor is like a NumPy array with two extra powers: it can live on a GPU, and it can track gradients.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])           # from a Python list
b = torch.zeros(2, 3)                        # 2 rows, 3 columns of zeros
c = torch.ones(2, 3)
d = torch.randn(2, 3)                        # samples from a standard normal distribution
e = torch.arange(0, 10)                      # 0..9
f = torch.linspace(0, 1, steps=5)            # 5 evenly spaced points from 0 to 1

for name, t in [("a", a), ("b", b), ("d", d), ("e", e), ("f", f)]:
    print(f"{name}: shape={tuple(t.shape)} dtype={t.dtype}\n{t}\n")

Notice `e` is `int64` while the others are `float32`. Neural networks train in floating point, and mixing dtypes is a common source of errors. You'll also see `float16` and `bfloat16` later: half-precision formats that LLMs use to cut memory in half.

In [ ]:
x = torch.arange(6)
print(x.dtype, "->", x.float().dtype, "->", x.to(torch.bfloat16).dtype)

# Bytes per element tells you memory cost. A 7B model in float32 vs bfloat16:
for dt in [torch.float32, torch.bfloat16, torch.int8]:
    bytes_per = torch.tensor([], dtype=dt).element_size()
    print(f"{str(dt):16} {bytes_per} bytes/param -> 7B params = {7e9 * bytes_per / 1e9:.0f} GB")

That last table is the same math behind quantization in module 01.

## 2. Shapes: indexing, reshaping, permuting

Think of a shape as nested boxes. Shape `(2, 3, 4)` means 2 blocks, each with 3 rows, each with 4 numbers. In LLM code you'll constantly see `(batch, sequence_length, hidden_dim)`, for example `(32, 1024, 4096)`.

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)
print(t)
print("shape:", t.shape)
print()
print("t[0]        ->", t[0].shape)         # first block
print("t[0, 1]     ->", t[0, 1].shape)      # first block, second row
print("t[:, 0, :]  ->", t[:, 0, :].shape)   # first row of every block
print("t[..., -1]  ->", t[..., -1].shape)   # last column everywhere ("..." means all leading dims)

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)

print(t.reshape(6, 4).shape)        # same 24 numbers, new shape
print(t.reshape(-1).shape)          # -1 means "figure this dim out" -> flatten
print(t.reshape(2, -1).shape)       # keep batch dim, flatten the rest

print(t.permute(2, 0, 1).shape)     # reorder dimensions: new order is old dims (2, 0, 1)
print(t.transpose(1, 2).shape)      # swap two dims

print(t.unsqueeze(0).shape)         # add a size-1 dim at the front
print(t.unsqueeze(0).squeeze(0).shape)

`reshape` keeps the numbers in the same order and just regroups them. `permute` and `transpose` actually reorder which axis is which. Mixing those two up silently scrambles your data, and it's one of the most common bugs in model code.

## 3. Broadcasting

When you combine tensors of different shapes, PyTorch stretches size-1 dimensions to match, without copying memory. The rule: line the shapes up from the right. Each pair of dims must be equal, or one of them must be 1 (or missing).

In [ ]:
M = torch.ones(4, 3)
row = torch.tensor([10.0, 20.0, 30.0])        # shape (3,)  -> treated as (1, 3)
col = torch.tensor([[1.0], [2.0], [3.0], [4.0]])  # shape (4, 1)

print("M + row:\n", M + row)                  # adds row to every row
print("M + col:\n", M + col)                  # adds col to every column
print("row + col shape:", (row + col).shape)   # (1,3) and (4,1) broadcast to (4,3)

try:
    M + torch.ones(4)    # (4,3) and (4,) -> trailing dims 3 vs 4 don't match
except RuntimeError as err:
    print("\nError, as expected:", str(err)[:80])

## 4. Matrix multiplication

`@` is matrix multiplication. It's almost all of the compute in a neural network. The rule: `(n, k) @ (k, m) -> (n, m)`. The inner dimensions must match.

In [ ]:
X = torch.randn(32, 784)     # a batch of 32 flattened 28x28 images
W = torch.randn(784, 128)    # weights of a layer mapping 784 inputs to 128 outputs
b = torch.randn(128)

out = X @ W + b              # this one line IS a fully connected layer
print(out.shape)

# Batched matmul: leading dims are treated as a batch. This is what attention does.
Q = torch.randn(8, 16, 64)   # (batch, seq, dim)
K = torch.randn(8, 16, 64)
scores = Q @ K.transpose(-2, -1)   # (8,16,64) @ (8,64,16) -> (8,16,16)
print(scores.shape)

That `scores` line is the core of self-attention in a transformer. Every token computes a similarity score with every other token. You'll build the rest of it in module 06.

## 5. Using your Mac's GPU

Tensors live on a device, and every tensor in an operation must be on the same device. Move them with `.to(device)`.

GPUs only win when there's enough work to spread across their cores. Watch the crossover.

In [ ]:
def sync():
    # GPU work is asynchronous. Without syncing, the timer would stop before the GPU finished.
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

def time_matmul(n, dev, repeats=5):
    A = torch.randn(n, n, device=dev)
    B = torch.randn(n, n, device=dev)
    A @ B; sync()                        # warm-up run, not timed
    start = time.perf_counter()
    for _ in range(repeats):
        A @ B
    sync()
    return (time.perf_counter() - start) / repeats

for n in [64, 512, 2048, 4096]:
    cpu_t = time_matmul(n, torch.device("cpu"))
    line = f"n={n:5}  cpu {cpu_t*1000:8.2f} ms"
    if device.type != "cpu":
        gpu_t = time_matmul(n, device)
        line += f"   {device.type} {gpu_t*1000:8.2f} ms   speedup {cpu_t/gpu_t:5.1f}x"
    print(line)

In [ ]:
cpu_tensor = torch.ones(3)
gpu_tensor = torch.ones(3, device=device)

try:
    cpu_tensor + gpu_tensor
    print("Same device (you're on CPU), so no error here.")
except RuntimeError as err:
    print("Error:", str(err)[:100])

print(cpu_tensor.to(device) + gpu_tensor)

Open Activity Monitor > Window > GPU History and rerun the benchmark cell with `n=4096` to watch your GPU spike.

## 6. Autograd

Training means nudging every weight in the direction that lowers the loss. To do that you need the gradient: how much the loss changes when each weight changes a tiny bit. Autograd computes all of those for you.

Take `y = x² + 3x`. By calculus, `dy/dx = 2x + 3`, so at `x = 2` the gradient is 7.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)   # "track operations on x"
y = x**2 + 3*x
y.backward()                                # compute dy/dx and store it in x.grad
print("y =", y.item(), "  dy/dx =", x.grad.item())

Three behaviors that cause real bugs:

In [ ]:
# 1. Gradients ACCUMULATE. Calling backward twice adds, it doesn't overwrite.
x = torch.tensor(2.0, requires_grad=True)
(x**2 + 3*x).backward()
(x**2 + 3*x).backward()
print("After two backward calls:", x.grad.item(), "(7 + 7)")
x.grad.zero_()
print("After zero_():", x.grad.item())

# 2. Inside torch.no_grad(), nothing is tracked. Use it for inference and weight updates.
with torch.no_grad():
    z = x * 10
print("z.requires_grad:", z.requires_grad)

# 3. .detach() gives a tensor cut off from the graph. .item() gives a plain Python number.
print(type(y.detach()), type(y.item()))

## 7. Gradient descent by hand

Now use gradients to learn. You'll fit a line `y = w*x + b` to noisy data, with no `nn` module and no optimizer, so you can see every step. The true values are `w = 2.5`, `b = -1.0`. The model starts knowing nothing.

In [ ]:
torch.manual_seed(0)
X = torch.linspace(-3, 3, 100)
Y = 2.5 * X - 1.0 + 0.5 * torch.randn(100)     # true line plus noise

w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.05

for step in range(101):
    pred = w * X + b                           # 1. forward pass
    loss = ((pred - Y) ** 2).mean()            # 2. loss (mean squared error)
    loss.backward()                            # 3. backward pass: compute d(loss)/dw and d(loss)/db
    with torch.no_grad():                      # 4. update: step downhill
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()             # 5. reset gradients for the next step
    if step % 20 == 0:
        print(f"step {step:3}  loss {loss.item():.4f}  w {w.item():.3f}  b {b.item():.3f}")

Those five numbered steps are training. Every model you'll ever train, including GPT-scale ones, runs this same loop. The difference is just a bigger model, more data, and a smarter update rule.

## 8. The same thing, the PyTorch way

In practice you use `nn.Module` for the model, a loss function from `nn`, and an optimizer that handles the update and zeroing.

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)          # holds w and b for you
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

Xc, Yc = X.unsqueeze(1), Y.unsqueeze(1)   # nn layers expect (batch, features): (100,) -> (100, 1)

for step in range(101):
    pred = model(Xc)
    loss = loss_fn(pred, Yc)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(f"step {step:3}  loss {loss.item():.4f}")

print("Learned w:", model.weight.item(), " b:", model.bias.item())

## 9. A real (small) neural network

A straight line can't fit a curve. Stack linear layers with a nonlinearity between them and the network can fit almost anything. Here it learns `sin(x)`, and it trains on your GPU.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
X = torch.linspace(-6, 6, 512).unsqueeze(1).to(device)
Y = torch.sin(X)

net = nn.Sequential(
    nn.Linear(1, 64),
    nn.Tanh(),
    nn.Linear(64, 64),
    nn.Tanh(),
    nn.Linear(64, 1),
).to(device)

print(net)
print("Parameters:", sum(p.numel() for p in net.parameters()))

opt = torch.optim.Adam(net.parameters(), lr=1e-2)   # Adam: the default optimizer for most deep learning
losses = []
for step in range(1500):
    loss = nn.functional.mse_loss(net(X), Y)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(losses); ax1.set_yscale("log"); ax1.set_title("Training loss"); ax1.set_xlabel("step")
with torch.no_grad():
    ax2.plot(X.cpu(), Y.cpu(), label="sin(x)")
    ax2.plot(X.cpu(), net(X).cpu(), "--", label="network")
ax2.legend(); ax2.set_title("Fit")
plt.show()

Try this: delete both `nn.Tanh()` lines and rerun. Without nonlinearities, stacked linear layers collapse into one linear function, so the network can only draw a straight line. That's why activation functions exist.

---
# Exercises

**Exercise 1: Shapes.** Create `t = torch.arange(24)`. Without running code first, predict the shape of each, then check:
- `t.reshape(2, 3, 4).permute(1, 2, 0)`
- `t.reshape(4, -1)[1:3]`
- `t.reshape(2, 3, 4)[:, None, :, 0]` (`None` inserts a new axis)

**Exercise 2: Normalize columns.** Given `data = torch.randn(100, 5) * 10 + 3`, use broadcasting (no loops) to make every column have mean 0 and standard deviation 1. Verify with `.mean(dim=0)` and `.std(dim=0)`.

**Exercise 3: Softmax from scratch.** Softmax turns a vector of scores into probabilities: `exp(x_i) / sum(exp(x_j))`. Write `my_softmax(x)` that works on the last dimension of any shape, and check it against `torch.softmax(x, dim=-1)`. Then try `torch.tensor([1000.0, 1001.0, 1002.0])`. Your version probably returns `nan`. Fix it by subtracting the max first, and explain to yourself why that doesn't change the answer.

**Exercise 4: Autograd check.** For `f(x) = x³ - 4x` at `x = 3`, compute the derivative by hand, then confirm with autograd.

**Exercise 5: Fit a curve by hand.** Data: `X = torch.linspace(-2, 2, 200)`, `Y = 3*X**2 - 2*X + 1 + 0.3*torch.randn(200)`. Use raw tensors `a, b, c` with `requires_grad=True` and the manual 5-step loop from section 7 to recover the coefficients.

---
# Solutions

In [ ]:
# Exercise 1
t = torch.arange(24)
print(t.reshape(2, 3, 4).permute(1, 2, 0).shape)   # (3, 4, 2)
print(t.reshape(4, -1)[1:3].shape)                 # (2, 6)
print(t.reshape(2, 3, 4)[:, None, :, 0].shape)     # (2, 1, 3)

In [ ]:
# Exercise 2
data = torch.randn(100, 5) * 10 + 3
normalized = (data - data.mean(dim=0)) / data.std(dim=0)   # (100,5) op (5,) broadcasts across rows
print(normalized.mean(dim=0))
print(normalized.std(dim=0))

In [ ]:
# Exercise 3
def my_softmax_naive(x):
    e = torch.exp(x)
    return e / e.sum(dim=-1, keepdim=True)   # keepdim keeps shape (..., 1) so broadcasting lines up

def my_softmax(x):
    x = x - x.max(dim=-1, keepdim=True).values   # exp(x - c) / sum(exp(x - c)) = exp(x) / sum(exp(x))
    e = torch.exp(x)
    return e / e.sum(dim=-1, keepdim=True)

x = torch.randn(4, 10)
print(torch.allclose(my_softmax(x), torch.softmax(x, dim=-1)))

big = torch.tensor([1000.0, 1001.0, 1002.0])
print("naive:", my_softmax_naive(big))   # exp(1000) overflows float32 -> inf/inf = nan
print("stable:", my_softmax(big))

In [ ]:
# Exercise 4: f'(x) = 3x^2 - 4 -> at x = 3: 27 - 4 = 23
x = torch.tensor(3.0, requires_grad=True)
(x**3 - 4*x).backward()
print(x.grad.item())

In [ ]:
# Exercise 5
torch.manual_seed(0)
X = torch.linspace(-2, 2, 200)
Y = 3*X**2 - 2*X + 1 + 0.3*torch.randn(200)

a = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
c = torch.zeros(1, requires_grad=True)
lr = 0.05

for step in range(2001):
    loss = ((a*X**2 + b*X + c - Y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in (a, b, c):
            p -= lr * p.grad
            p.grad.zero_()
    if step % 500 == 0:
        print(f"step {step:4}  loss {loss.item():.4f}  a {a.item():.3f}  b {b.item():.3f}  c {c.item():.3f}")